# 3 · Create the Supervisor Agent

Creates a **Supervisor Agent** (multi-agent system) that routes questions
between two sub-agents:

1. **Contracts search** — the **Vector Search index** from notebook `1`, attached
   directly as a native `vector_search_index` sub-agent. This replaces the
   Knowledge Assistant, which is not available on Databricks Free Edition.
2. **Genie Space** — answers structured analytics questions about marketing campaign sales
   and marketing data.

**What this notebook does**
1. Creates (or reuses) the Supervisor Agent
2. Attaches the Vector Search index and the Genie Space as tools
3. Waits for the supervisor endpoint to be ready

> The `vector_search_index` tool type is available through the Supervisor Agents
> REST API. We call it directly so the notebook works regardless of the installed
> `databricks-sdk` version.

### Install Dependencies

In [ ]:
%pip install --upgrade databricks-sdk
dbutils.library.restartPython()

### Define Parameters
Review the widgets before running. Defaults match the **Marketing Campaign Analytics** layout.
| Widget | Default | What it controls |
| --- | --- | --- |
| `CATALOG_NAME` | `workspace` | Unity Catalog for demo objects. Default `workspace` is pre-provisioned — no catalog creation needed. |
| `SCHEMA_NAME` | `marketing_campaign` | Schema inside the catalog for sales data and contracts. |
| `INDEX_NAME` | `contract_chunks_index` | Vector Search index tool for contracts (from notebook `1`). |
| `GENIE_SPACE_ID` | *(empty)* | Optional. If blank, the space is auto-found from `GENIE_SPACE_TITLE` below — no copy-paste needed. |
| `GENIE_SPACE_TITLE` | `Marketing Campaign Analytics — Genie Space` | Title of the Genie space created in `2_genie/0`. Used to auto-resolve `GENIE_SPACE_ID`. |
| `SUPERVISOR_NAME` | `Marketing Campaign Insights Supervisor` | Display name of the Supervisor Agent. |
| `SUPERVISOR_DESCRIPTION` | *(see default)* | Short description shown in Agent Bricks. |
| `SUPERVISOR_INSTRUCTIONS` | *(see default)* | Routing instructions for contracts vs. analytics. |


In [ ]:
dbutils.widgets.text("CATALOG_NAME", "workspace", "CATALOG_NAME — Unity Catalog catalog for demo objects")
dbutils.widgets.text("SCHEMA_NAME", "marketing_campaign", "SCHEMA_NAME — schema inside the catalog")
dbutils.widgets.text("INDEX_NAME", "contract_chunks_index", "INDEX_NAME — Vector Search index (from notebook 1)")
dbutils.widgets.text("GENIE_SPACE_ID", "", "GENIE_SPACE_ID — optional; auto-resolved from GENIE_SPACE_TITLE if left blank")
dbutils.widgets.text("GENIE_SPACE_TITLE", "Marketing Campaign Analytics — Genie Space", "GENIE_SPACE_TITLE — title used in 2_genie/0 (used to auto-find the space)")
dbutils.widgets.text("SUPERVISOR_NAME", "Marketing Campaign Insights Supervisor", "SUPERVISOR_NAME — display name for the Supervisor Agent")
dbutils.widgets.text(
    "SUPERVISOR_DESCRIPTION",
    "Unified assistant for marketing campaign analytics: answers contract-document questions and structured sales/marketing analytics questions.",
    "SUPERVISOR_DESCRIPTION — short Agent Bricks description",
)
dbutils.widgets.text(
    "SUPERVISOR_INSTRUCTIONS",
    "Route document and contract questions (terms, parties, clauses, obligations) to the contracts vector search index. Route data and analytics questions (sales, revenue, customers, products, stores, campaigns) to the sales analytics Genie space. If a question spans both, gather information from each and combine the answers.",
    "SUPERVISOR_INSTRUCTIONS — routing rules for tools",
)

### Read Parameters

The Genie space id is resolved in the next step (auto-found from `GENIE_SPACE_TITLE`,
or taken from `GENIE_SPACE_ID` if you pasted one). No manual copy-paste required.

In [ ]:
CATALOG_NAME = dbutils.widgets.get("CATALOG_NAME")
SCHEMA_NAME = dbutils.widgets.get("SCHEMA_NAME")
INDEX_NAME = dbutils.widgets.get("INDEX_NAME")
GENIE_SPACE_ID = dbutils.widgets.get("GENIE_SPACE_ID").strip()
GENIE_SPACE_TITLE = dbutils.widgets.get("GENIE_SPACE_TITLE").strip()
SUPERVISOR_NAME = dbutils.widgets.get("SUPERVISOR_NAME")
SUPERVISOR_DESCRIPTION = dbutils.widgets.get("SUPERVISOR_DESCRIPTION")
SUPERVISOR_INSTRUCTIONS = dbutils.widgets.get("SUPERVISOR_INSTRUCTIONS")

INDEX_FQN = f"{CATALOG_NAME}.{SCHEMA_NAME}.{INDEX_NAME}"
print("Index           :", INDEX_FQN)
print("Genie space id  :", GENIE_SPACE_ID or "(will auto-resolve by title)")
print("Genie space ttl :", GENIE_SPACE_TITLE)

### Initialize the Workspace Client

In [ ]:
from databricks.sdk import WorkspaceClient
from databricks.sdk.service.supervisoragents import SupervisorAgent, Tool, GenieSpace

w = WorkspaceClient()

### Resolve the Genie Space

If you pasted a `GENIE_SPACE_ID`, that value is used directly. Otherwise the
notebook lists your Genie spaces and picks the one whose title matches
`GENIE_SPACE_TITLE` (the default from `2_genie/0`). This avoids having to
copy-paste any id between notebooks.

In [ ]:
if not GENIE_SPACE_ID:
    spaces = []
    page_token = None
    while True:
        resp = w.genie.list_spaces(page_size=100, page_token=page_token)
        page = list(getattr(resp, "spaces", []) or [])
        spaces.extend(page)
        page_token = getattr(resp, "next_page_token", None)
        if not page_token:
            break
    match = next((s for s in spaces if (getattr(s, "title", "") or "") == GENIE_SPACE_TITLE), None)
    assert match is not None, (
        f"No Genie space found with title '{GENIE_SPACE_TITLE}'. "
        "Run 2_genie/0_CreateGenieSpace first, or paste the space id into the GENIE_SPACE_ID widget."
    )
    GENIE_SPACE_ID = match.space_id
    print(f"Auto-resolved Genie space '{GENIE_SPACE_TITLE}' -> {GENIE_SPACE_ID}")
else:
    print("Using GENIE_SPACE_ID from widget:", GENIE_SPACE_ID)

assert GENIE_SPACE_ID, "GENIE_SPACE_ID is still empty."
print("Genie space id :", GENIE_SPACE_ID)

### Create (or Reuse) the Supervisor Agent

In [ ]:
existing = next(
    (s for s in w.supervisor_agents.list_supervisor_agents() if s.display_name == SUPERVISOR_NAME),
    None,
)
if existing:
    sup = existing
    print(f'Reusing existing Supervisor Agent: {sup.name}')
else:
    sup = w.supervisor_agents.create_supervisor_agent(
        supervisor_agent=SupervisorAgent(
            display_name=SUPERVISOR_NAME, description=SUPERVISOR_DESCRIPTION, instructions=SUPERVISOR_INSTRUCTIONS,
        )
    )
    print(f'Created Supervisor Agent: {sup.name}')

SUP_RESOURCE_NAME = sup.name  # supervisor-agents/{id}
print('Resource name:', SUP_RESOURCE_NAME)
print('Endpoint name:', sup.endpoint_name)

### Add Tools: Vector Search Index + Genie Space

The Vector Search index is added via a direct REST call because the
`vector_search_index` tool type may not yet be modeled in every SDK version.

In [ ]:
existing_tools = {t.tool_id for t in w.supervisor_agents.list_tools(parent=SUP_RESOURCE_NAME)}

# Contracts: native Vector Search index sub-agent (REST)
if 'contracts_search' in existing_tools:
    print("Tool 'contracts_search' already present.")
else:
    body = {
        'tool_type': 'vector_search_index',
        'description': 'Searches marketing campaign contract documents (parties, terms, pricing, dates, obligations, clauses) via Vector Search.',
        'vector_search_index': {'name': INDEX_FQN, 'columns': ['source_uri', 'chunk_to_retrieve']},
    }
    w.api_client.do('POST', f'/api/2.1/{SUP_RESOURCE_NAME}/tools',
                    query={'tool_id': 'contracts_search'}, body=body,
                    headers={'Accept': 'application/json', 'Content-Type': 'application/json'})
    print("Added tool 'contracts_search' (vector_search_index).")

# Analytics: Genie space (SDK)
if 'sales_analytics_genie' in existing_tools:
    print("Tool 'sales_analytics_genie' already present.")
else:
    w.supervisor_agents.create_tool(
        parent=SUP_RESOURCE_NAME, tool_id='sales_analytics_genie',
        tool=Tool(
            tool_type='genie_space',
            description='Answers structured analytics questions about marketing campaign sales and marketing: revenue, transactions, customers, products, stores, campaigns, and salespeople.',
            genie_space=GenieSpace(id=GENIE_SPACE_ID),
        ),
    )
    print("Added tool 'sales_analytics_genie' (genie_space).")

print('Tools:', [(t.tool_id, t.tool_type) for t in w.supervisor_agents.list_tools(parent=SUP_RESOURCE_NAME)])

### Wait for the Supervisor Endpoint to be Ready

In [ ]:
import time
sup = w.supervisor_agents.get_supervisor_agent(name=SUP_RESOURCE_NAME)
ENDPOINT_NAME = sup.endpoint_name
print('Supervisor endpoint:', ENDPOINT_NAME)

deadline = time.time() + 60 * 20
while True:
    ep = w.serving_endpoints.get(ENDPOINT_NAME)
    ready = ep.state.ready if ep.state else None
    cu = ep.state.config_update if ep.state else None
    print(f'ready={ready} config_update={cu}')
    if ready and 'READY' in str(ready) and (cu is None or 'IN_PROGRESS' not in str(cu)):
        break
    if time.time() > deadline:
        print('Endpoint still updating; continue once it reports READY.'); break
    time.sleep(30)

### Result

In [ ]:
print('Supervisor Agent name :', SUPERVISOR_NAME)
print('Supervisor Agent id   :', sup.supervisor_agent_id)
print('Serving endpoint      :', sup.endpoint_name)
print('\nUse SUPERVISOR_NAME in notebook 4 to query it.')